In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=LRmZZu7l2lnQkxuTcKgKBM0EXWIfjD&access_type=offline&code_challenge=HpCGw3Fy4kOELpLOBZbGsyP8z9iSscEF9huVzckxBEU&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/21 10:59:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/21 10:59:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
path_to_release_folder="gs://open-targets-data-releases/25.06/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

sl_eff=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect")

l2g_full=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet")

In [4]:
x=session.spark.read.parquet("gs://ot-team/jroldan/analysis/goldStandard_20241031/goldStandardDrugs_outer.parquet/")

In [5]:
x.show()

+---------------+--------------+-------------+---------+------------------+------------+--------------+------------------+----------------------------+-------------+----------------------+----------------------------+
|       targetId|approvedSymbol|    diseaseId|unionMesh|          approved|maxClinPhase|phasesApproved|combined_max_phase|transformed_combinedMaxPhase|hasNelsonData|differencesNoDrugLabel|differencesApprovedDrugLabel|
+---------------+--------------+-------------+---------+------------------+------------+--------------+------------------+----------------------------+-------------+----------------------+----------------------------+
|ENSG00000183044|          ABAT|  EFO_0002610|     NULL|              NULL|         2.0|           2.0|              NULL|                        NULL|       noMesh|                  NULL|                        NULL|
|ENSG00000183044|          ABAT|MONDO_0100062|     NULL|approvedIndication|         3.0|           4.0|              NULL|      

In [6]:
x.groupBy("hasNelsonData").count().show()

+-------------+-----+
|hasNelsonData|count|
+-------------+-----+
| noNelsonData|43959|
|       noMesh|31359|
|          yes|25894|
+-------------+-----+



In [7]:
x=x.filter(f.col("hasNelsonData")=="yes").cache()
x.count()

25894

In [8]:
x=x.filter(f.col("targetId").isNotNull()).cache()
x.count()

6047

In [9]:
x.withColumn("labelMatch", f.when(f.col("differencesNoDrugLabel")==f.col("differencesApprovedDrugLabel"), 1).otherwise(0)).groupBy("labelMatch").count().show()

+----------+-----+
|labelMatch|count|
+----------+-----+
|         1| 6026|
|         0|   21|
+----------+-----+



In [10]:
x.groupBy("differencesNoDrugLabel").count().show()

+----------------------+-----+
|differencesNoDrugLabel|count|
+----------------------+-----+
|                 equal| 2551|
|        Nelson<<Chembl| 2539|
|        Nelson>>Chembl|  957|
+----------------------+-----+



In [11]:
x.show()

+---------------+--------------+-------------+---------+------------------+------------+--------------+------------------+----------------------------+-------------+----------------------+----------------------------+
|       targetId|approvedSymbol|    diseaseId|unionMesh|          approved|maxClinPhase|phasesApproved|combined_max_phase|transformed_combinedMaxPhase|hasNelsonData|differencesNoDrugLabel|differencesApprovedDrugLabel|
+---------------+--------------+-------------+---------+------------------+------------+--------------+------------------+----------------------------+-------------+----------------------+----------------------------+
|ENSG00000183044|          ABAT|  EFO_0000474|  D004827|approvedIndication|         4.0|           4.0|       Preclinical|                         0.5|          yes|        Nelson<<Chembl|              Nelson<<Chembl|
|ENSG00000183044|          ABAT|  EFO_0004263|  D004828|approvedIndication|         4.0|           4.0|       Preclinical|      

In [12]:
x.groupBy("transformed_combinedMaxPhase").count().show()

+----------------------------+-----+
|transformed_combinedMaxPhase|count|
+----------------------------+-----+
|                         1.0|  885|
|                         4.0| 1018|
|                         0.5| 1168|
|                         3.0|  697|
|                         2.0| 2279|
+----------------------------+-----+



In [13]:
disease_index_path=path_to_release_folder+"output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path=path_to_release_folder+"output/evidence/sourceId=chembl"
chembl_evidence=session.spark.read.parquet(platform_chembl_evidence_path)

In [15]:
x=x.withColumnRenamed("transformed_combinedMaxPhase", "clinicalPhase").select("diseaseId", "clinicalPhase","targetId").cache()
x.count()

6047

In [17]:
x.select("diseaseId","targetId").distinct().count()

6047

In [18]:
x.write.mode("overwrite").parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/pharmacoproject_processed.parquet")
